# 🛍️ Laboratorio: K-Means con el caso Mall Customers

## 📖 La historia
Un centro comercial lanzó una **tarjeta de membresía**. Al registrarse, cada cliente dio su género, edad e ingreso anual; con el uso de la tarjeta, el centro comercial le asignó un **puntaje de gasto** de 1 a 100 según qué tanto y qué tan seguido compra.

Hoy hay **200 clientes** con tarjeta. El área de marketing tiene presupuesto para **una campaña distinta por segmento**, pero nadie sabe cuántos segmentos hay ni cómo son. Nadie etiquetó a los clientes: no hay "respuesta correcta" contra la cual comparar. Es un problema **no supervisado**.

> ### ❓ ¿Qué segmentos de clientes tiene el centro comercial y qué campaña le toca a cada uno?

## 🧭 Lo que vas a aprender
1. Explorar los datos buscando **grupos naturales** antes de modelar
2. Entender **por qué escalar** antes de usar distancias
3. Elegir el número de clusters **k** con varias señales: **codo, silhouette y gráfica de silueta**
4. **Perfilar** cada cluster en unidades originales y ponerle **nombre de negocio**
5. Decidir **qué variables** entran al modelo (2D vs 3D, ¿y el género?) y traducir los segmentos en **acciones**

## ⏱️ Agenda (3 horas)
| Parte | Tema | Tiempo |
|---|---|---|
| 0 | Preparación | 10 min |
| 1 | Conocer los datos | 15 min |
| 2 | Explorar visualmente | 20 min |
| 3 | ¿Por qué escalar? | 15 min |
| 4 | Modelo 2D: ingreso y gasto, ¿cuántos clusters? | 30 min |
| ☕ | Descanso | 10 min |
| 5 | Interpretar los segmentos | 25 min |
| 6 | Modelo 3D: agregamos la edad | 25 min |
| 7 | ¿Y el género? | 10 min |
| 8 | Campañas y conclusiones | 20 min |

## 📋 Las variables
| Variable | Descripción |
|---|---|
| `cliente_id` | Identificador del cliente (no se usa para agrupar) |
| `genero` | `Female` / `Male` |
| `edad` | Edad en años |
| `ingreso_anual` | Ingreso anual en **miles de dólares** (k$) |
| `gasto` | Puntaje de gasto de **1 a 100** asignado por el centro comercial (más alto = compra más) |

---
# 0️⃣ Preparación (10 min)

Ejecuta las siguientes celdas **una sola vez**. Contienen funciones que te ahorran código repetitivo para que te concentres en **entender** los resultados.

## 🛠️ Funciones auxiliares de visualización

Ejecuta la siguiente celda **una sola vez** al inicio. Después sólo tienes que **llamar** a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `plot_distributions(df, columnas)` | Histograma + boxplot de variables numéricas |
| `plot_correlation_matrix(df, columnas)` | Matriz de correlación |
| `plot_pairplot(df, columnas, color=None)` | Dispersión entre todas las variables numéricas |
| `plot_clusters(df, x, y, labels, centroides=None)` | Puntos en 2D coloreados por cluster (el ruido en gris) |
| `plot_clusters_3d(df, x, y, z, labels)` | Lo mismo en 3D (se puede rotar) |
| `plot_codo(ks, inercias)` | Método del codo para elegir k |
| `plot_silueta_k(tabla)` | Silhouette promedio según k |
| `plot_silueta(X, labels)` | Coeficiente de silueta de cada punto, por cluster |
| `plot_k_distancia(X, k, eps=None)` | Distancia al k-ésimo vecino: ayuda a elegir `eps` de DBSCAN |
| `plot_varianza_explicada(pca)` | Varianza explicada por componente y acumulada |
| `plot_cargas(pca, columnas)` | Peso de cada variable original en cada componente |
| `plot_biplot(scores, pca, columnas, labels=None)` | Puntos en PC1-PC2 con una flecha por variable |

In [9]:
# Funciones auxiliares de visualización
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _etiquetas_texto(labels):
    """Convierte etiquetas de cluster a texto para que plotly use colores discretos; -1 = 'Ruido'."""
    labels = np.asarray(labels)
    return np.where(labels == -1, 'Ruido', np.char.add('Cluster ', labels.astype(str)))


def _orden_clusters(texto):
    """Orden de la leyenda: Cluster 0, 1, 2, … y 'Ruido' al final."""
    unicos = sorted(set(texto), key=lambda t: (t == 'Ruido', int(t.split()[-1]) if t != 'Ruido' else 0))
    return {'cluster': unicos}


def plot_distributions(df, columns, nbins=30):
    """Histograma con boxplot marginal para cada variable numérica."""
    for col in columns:
        fig = px.histogram(
            df,
            x=col,
            nbins=nbins,
            marginal='box',
            opacity=0.7,
            title=f'Distribución de {col}'
        )
        fig.update_layout(bargap=0.2)
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def plot_pairplot(df, columns, color=None):
    """Matriz de dispersión (pairplot) entre las variables numéricas."""
    fig = px.scatter_matrix(
        df,
        dimensions=columns,
        color=color,
        title='Pairplot de Variables Numéricas'
    )
    fig.update_layout(width=1000, height=1000, title_font_size=20)
    fig.update_traces(diagonal_visible=True, marker={'size': 4, 'opacity': 0.6})
    fig.show()


def plot_clusters(df, x, y, labels, centroides=None, title='Clusters'):
    """Dispersión 2D coloreada por cluster. El ruido de DBSCAN (-1) aparece en gris.
    centroides: arreglo (k, 2) en las mismas unidades que x, y (opcional)."""
    datos = df[[x, y]].copy()
    datos['cluster'] = _etiquetas_texto(labels)
    fig = px.scatter(
        datos,
        x=x,
        y=y,
        color='cluster',
        category_orders=_orden_clusters(datos['cluster']),
        color_discrete_map={'Ruido': 'lightgray'},
        opacity=0.8,
        title=title
    )
    if centroides is not None:
        centroides = np.asarray(centroides)
        fig.add_trace(go.Scatter(
            x=centroides[:, 0],
            y=centroides[:, 1],
            mode='markers',
            marker={'symbol': 'x', 'size': 16, 'color': 'black', 'line': {'width': 2}},
            name='Centroides'
        ))
    fig.update_layout(width=800, height=550)
    fig.show()


def plot_clusters_3d(df, x, y, z, labels, title='Clusters en 3D'):
    """Dispersión 3D coloreada por cluster (se puede rotar con el mouse)."""
    datos = df[[x, y, z]].copy()
    datos['cluster'] = _etiquetas_texto(labels)
    fig = px.scatter_3d(
        datos,
        x=x,
        y=y,
        z=z,
        color='cluster',
        category_orders=_orden_clusters(datos['cluster']),
        color_discrete_map={'Ruido': 'lightgray'},
        opacity=0.8,
        title=title
    )
    fig.update_traces(marker={'size': 4})
    fig.update_layout(width=850, height=650)
    fig.show()


def plot_codo(ks, inercias):
    """Método del codo: inercia (suma de distancias² al centroide) para cada k."""
    fig = px.line(
        x=list(ks),
        y=list(inercias),
        markers=True,
        labels={'x': 'Número de clusters (k)', 'y': 'Inercia'},
        title='Método del codo'
    )
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_silueta_k(tabla):
    """Silhouette promedio para cada k (tabla de tabla_metricas_k). Más alto es mejor."""
    fig = px.line(
        tabla,
        x='k',
        y='silhouette',
        markers=True,
        labels={'k': 'Número de clusters (k)', 'silhouette': 'Silhouette promedio'},
        title='Silhouette según k (↑ mejor)'
    )
    fig.update_xaxes(dtick=1)
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_silueta(X, labels, title=None):
    """Gráfica de silueta: el ancho de cada barra es el coeficiente de un punto, agrupado por cluster.
    La línea punteada es el promedio. Los puntos de ruido (-1) se excluyen."""
    from sklearn.metrics import silhouette_samples, silhouette_score

    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    X, labels = X[mask], labels[mask]
    valores = silhouette_samples(X, labels)
    promedio = silhouette_score(X, labels)

    fig = go.Figure()
    y_inf = 10
    for c in np.unique(labels):
        v = np.sort(valores[labels == c])
        y = np.arange(y_inf, y_inf + len(v))
        fig.add_trace(go.Scatter(
            x=v, y=y, fill='tozerox', mode='lines', line={'width': 0.5}, name=f'Cluster {c}'
        ))
        y_inf += len(v) + 10
    fig.add_vline(x=promedio, line_dash='dash', line_color='red',
                  annotation_text=f'promedio = {promedio:.3f}', annotation_position='top')
    fig.update_layout(
        title=title or f'Gráfica de silueta (k = {len(np.unique(labels))})',
        xaxis_title='Coeficiente de silueta',
        yaxis={'showticklabels': False, 'title': 'Puntos (ordenados por cluster)'},
        width=750,
        height=500
    )
    fig.show()


def plot_k_distancia(X, k, eps=None):
    """Distancia que necesita cada punto para tener k vecinos (contándose a sí mismo), ordenada.
    Usa k = min_samples: igual que DBSCAN, el punto cuenta como uno de sus k vecinos, así que un punto
    es núcleo con un eps dado si su distancia en esta curva es <= eps. El 'codo' sugiere un buen eps.
    eps (opcional) dibuja una línea horizontal de referencia."""
    from sklearn.neighbors import NearestNeighbors

    # n_neighbors=k incluye al propio punto (distancia 0), como min_samples en DBSCAN
    vecinos = NearestNeighbors(n_neighbors=k).fit(X)
    distancias, _ = vecinos.kneighbors(X)
    d_k = np.sort(distancias[:, -1])

    fig = px.line(
        x=np.arange(len(d_k)),
        y=d_k,
        labels={'x': 'Puntos ordenados por distancia', 'y': f'Distancia para tener {k} vecinos (incluido él)'},
        title=f'Gráfica de k-distancia (k = {k})'
    )
    if eps is not None:
        fig.add_hline(y=eps, line_dash='dash', line_color='red',
                      annotation_text=f'eps = {eps}', annotation_position='top left')
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_varianza_explicada(pca):
    """Scree plot: varianza explicada por componente (barras) y acumulada (línea)."""
    ratio = pca.explained_variance_ratio_
    n = np.arange(1, len(ratio) + 1)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=n, y=ratio, name='Por componente'))
    fig.add_trace(go.Scatter(x=n, y=np.cumsum(ratio), mode='lines+markers', name='Acumulada'))
    for umbral in (0.80, 0.95):
        fig.add_hline(y=umbral, line_dash='dot', line_color='gray',
                      annotation_text=f'{umbral:.0%}', annotation_position='right')
    fig.update_layout(
        title='Varianza explicada por componente',
        xaxis={'title': 'Componente principal', 'dtick': 1 if len(ratio) <= 30 else None},
        yaxis={'title': 'Proporción de varianza', 'tickformat': '.0%', 'range': [0, 1.05]},
        width=800,
        height=450
    )
    fig.show()


def plot_cargas(pca, columnas, n_componentes=None):
    """Mapa de calor de las cargas (loadings): cuánto pesa cada variable original en cada componente."""
    n = n_componentes or pca.n_components_
    cargas = pd.DataFrame(
        pca.components_[:n].T,
        index=columnas,
        columns=[f'PC{i + 1}' for i in range(n)]
    ).round(2)
    fig = px.imshow(
        cargas,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        aspect='auto',
        title='Cargas de cada variable en los componentes'
    )
    fig.update_layout(width=200 + 90 * n, height=150 + 40 * len(columnas))
    fig.show()


def plot_biplot(scores, pca, columnas, labels=None, escala=None, title='Biplot PC1 vs PC2'):
    """Puntos en el plano PC1-PC2 más una flecha por variable original (sus cargas).
    scores: salida de pca.transform(X). labels (opcional) colorea los puntos por cluster o grupo."""
    scores = np.asarray(scores)
    datos = pd.DataFrame({'PC1': scores[:, 0], 'PC2': scores[:, 1]})
    if labels is not None:
        datos['grupo'] = _etiquetas_texto(labels) if np.issubdtype(np.asarray(labels).dtype, np.integer) \
            else np.asarray(labels).astype(str)
    fig = px.scatter(datos, x='PC1', y='PC2', color='grupo' if labels is not None else None,
                     opacity=0.6, title=title)
    cargas = pca.components_[:2].T
    escala = escala or 0.9 * np.abs(scores[:, :2]).max() / np.abs(cargas).max()
    for (cx, cy), nombre in zip(cargas * escala, columnas):
        fig.add_annotation(x=cx, y=cy, ax=0, ay=0, xref='x', yref='y', axref='x', ayref='y',
                           showarrow=True, arrowhead=2, arrowwidth=1.5, arrowcolor='black')
        fig.add_annotation(x=cx * 1.08, y=cy * 1.08, text=nombre, showarrow=False, font={'size': 12})
    r = pca.explained_variance_ratio_
    fig.update_layout(
        xaxis_title=f'PC1 ({r[0]:.1%} de la varianza)',
        yaxis_title=f'PC2 ({r[1]:.1%} de la varianza)',
        width=850,
        height=650
    )
    fig.show()
    


    

    

### 🧰 Funciones de clustering

Hacen en una línea lo que de otra forma repetirías con cada modelo:

| Función | ¿Para qué sirve? |
|---|---|
| `evaluar_clustering(nombre, X, labels)` | Número de clusters, % de ruido y silhouette (sin contar el ruido) |
| `tabla_metricas_k(X, ks)` | Ajusta K-Means para cada k y regresa inercia y silhouette en una tabla |
| `perfil_clusters(df, labels, columnas)` | Tamaño y promedio de cada variable por cluster |

In [10]:
# Funciones de clustering
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


def evaluar_clustering(nombre, X, labels):
    """Silhouette de un agrupamiento. El ruido (-1) se excluye del cálculo,
    pero se reporta qué porcentaje de los puntos quedó como ruido."""
    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    n_clusters = len(np.unique(labels[mask]))
    fila = {
        'modelo': nombre,
        'n_clusters': n_clusters,
        'pct_ruido': float(round(100 * (~mask).mean(), 1)),
    }
    if n_clusters >= 2:
        fila['silhouette'] = float(round(silhouette_score(X[mask], labels[mask]), 3))
    else:
        fila['silhouette'] = np.nan
    return fila


def tabla_metricas_k(X, ks, random_state=42):
    """Ajusta K-Means para cada k y regresa una tabla con la inercia (para el codo) y la silhouette."""
    filas = []
    for k in ks:
        modelo = KMeans(n_clusters=k, n_init=10, random_state=random_state).fit(X)
        fila = evaluar_clustering(f'k={k}', X, modelo.labels_)
        filas.append({'k': k, 'inercia': float(round(modelo.inertia_, 1)), 'silhouette': fila['silhouette']})
    return pd.DataFrame(filas)


def perfil_clusters(df, labels, columnas):
    """Promedio de cada variable por cluster, con tamaño y porcentaje. Sirve para ponerle nombre a cada grupo."""
    datos = df[columnas].copy()
    datos['cluster'] = np.asarray(labels)
    perfil = datos.groupby('cluster')[columnas].mean().round(1)
    perfil.insert(0, 'n', datos['cluster'].value_counts().sort_index())
    perfil.insert(1, 'pct', (100 * perfil['n'] / perfil['n'].sum()).round(1))
    perfil.index = ['Ruido' if c == -1 else f'Cluster {c}' for c in perfil.index]
    return perfil


### 📥 Cargar los datos

La siguiente celda define dos funciones que usaremos en este laboratorio **y en las siguientes sesiones** con este mismo caso:

| Función | ¿Qué hace? |
|---|---|
| `cargar_mall()` | Descarga los 200 clientes y renombra las columnas al español (`genero`, `edad`, `ingreso_anual`, `gasto`) |
| `preparar_mall(df, columnas)` | Estandariza las columnas elegidas (media 0, desviación 1) y regresa `X_scaled` y el `scaler` |

`COLS_MALL` es la lista de las tres variables numéricas: `['edad', 'ingreso_anual', 'gasto']`.

In [11]:
# Datos del caso Mall Customers
import pandas as pd
from sklearn.preprocessing import StandardScaler

URL_MALL = 'https://raw.githubusercontent.com/jorgermzg15/data-files/refs/heads/main/Mall_Customers.csv'
COLS_MALL = ['edad', 'ingreso_anual', 'gasto']


def cargar_mall():
    """Descarga los 200 clientes del centro comercial y renombra las columnas al español."""
    df = pd.read_csv(URL_MALL)
    return df.rename(columns={
        'CustomerID': 'cliente_id',
        'Genre': 'genero',
        'Age': 'edad',
        'Annual Income (k$)': 'ingreso_anual',
        'Spending Score (1-100)': 'gasto',
    })


def preparar_mall(df, columnas=COLS_MALL):
    """Estandariza las columnas elegidas (media 0, desviación 1) y regresa la matriz escalada y el scaler."""
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df[columnas])
    return X_scaled, scaler








In [12]:
df = cargar_mall()
df.head()


,cliente_id,genero,edad,ingreso_anual,gasto
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40


---
# 1️⃣ Conocer los datos (15 min)

Antes de agrupar hay que saber qué tenemos: cuántos clientes, qué tipo de dato es cada columna, si hay nulos o duplicados y en qué rangos se mueven las variables.

In [13]:
# ¿Cuántas filas y columnas? ¿Qué tipo de dato tiene cada columna?
# Pista: df.shape y df.dtypes
# ✍️ Tu código aquí
df = cargar_mall()

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])
print("\nTipo de dato de cada columna:")
print(df.dtypes)

Filas: 200
Columnas: 5

Tipo de dato de cada columna:
cliente_id       int64
genero             str
edad             int64
ingreso_anual    int64
gasto            int64
dtype: object


In [14]:
# ¿Hay nulos o clientes duplicados?
# Pista: df.duplicated().sum() y df.isna().sum()
# ✍️ Tu código aquí
print("Filas duplicadas:", df.duplicated().sum())
print("Identificadores de cliente repetidos:", df['cliente_id'].duplicated().sum())

print("\nValores nulos por columna:")
print(df.isna().sum())


Filas duplicadas: 0
Identificadores de cliente repetidos: 0

Valores nulos por columna:
cliente_id       0
genero           0
edad             0
ingreso_anual    0
gasto            0
dtype: int64


In [15]:
# Estadísticas descriptivas de las variables numéricas
# Pista: df[COLS_MALL].describe().round(1)
# ✍️ Tu código aquí
df[COLS_MALL].describe().round(1)

,edad,ingreso_anual,gasto
count,200.0,200.0,200.0
mean,38.8,60.6,50.2
std,14.0,26.3,25.8
min,18.0,15.0,1.0
25%,28.8,41.5,34.8
50%,36.0,61.5,50.0
75%,49.0,78.0,73.0
max,70.0,137.0,99.0


In [16]:
# ¿Cuántos clientes hay de cada género?
# Pista: df['genero'].value_counts()
# ✍️ Tu código aquí
df['genero'].value_counts()


genero
Female    112
Male       88
Name: count, dtype: int64

✍️ **¿Qué observas?** ¿Hay nulos o duplicados? ¿En qué rango se mueve cada variable? ¿Cuántas mujeres y hombres hay? ¿Por qué `cliente_id` no debería entrar al modelo?

RELLENAR CON TUS COMENTARIOS
-Tenemos 200 clientes, 112 mujeres y 88 hombres
-El rango de edad va desde los 70 anos
-El ingreso anual va desde $15 K USD a $137 K USD

---
# 2️⃣ Explorar visualmente (20 min)

En clustering, la exploración visual es todavía más importante que en regresión: buscamos **grupos que se vean a simple vista** y variables que separen a los clientes.

In [17]:
# Distribución de cada variable numérica
# Pista: plot_distributions(df, COLS_MALL)
# ✍️ Tu código aquí
plot_distributions(df, COLS_MALL)

In [18]:
# Correlación entre las variables numéricas
# Pista: plot_correlation_matrix(df, COLS_MALL)
# ✍️ Tu código aquí
plot_correlation_matrix(df, COLS_MALL)

In [19]:
# Esta celda va de regalo: léela con calma y ejecútala
# Pairplot: cada punto es un cliente, coloreado por género
plot_pairplot(df, COLS_MALL, color='genero')

In [20]:
# Esta celda va de regalo: léela con calma y ejecútala
# Zoom a la pareja más interesante: ingreso vs gasto
fig = px.scatter(
    df,
    x='ingreso_anual',
    y='gasto',
    opacity=0.7,
    labels={'ingreso_anual': 'Ingreso anual (k$)', 'gasto': 'Puntaje de gasto (1-100)'},
    title='Ingreso vs gasto: ¿cuántos grupos ves?'
)
fig.update_layout(width=750, height=550)
fig.show()

✍️ **¿Qué observas?**
- ¿Ingreso y gasto están correlacionados? ¿Qué significa eso para el negocio?
- ¿Qué relación hay entre la edad y el gasto?
- En la gráfica de ingreso vs gasto, ¿cuántos grupos ves a simple vista? ¿Dónde están?
- ¿Los colores por género forman grupos separados?

RELLENAR CON TUS COMENTARIOS
- No, no estan, ganar mas no significa gastar mas
- la edad tiene correlacion negativa de -0.33, los clientes jovenes tienden a gastar mas.
- 5 grupos, uno al centro y 4 en las esquinas.
-No tiene un rol relevante

---
# 3️⃣ ¿Por qué escalar? (15 min)

K-Means agrupa por **distancia euclidiana**. Si una variable tiene números mucho más grandes que otra, **domina la distancia** y la otra prácticamente no cuenta.

En nuestros datos, ingreso (desviación **26.3** k$) y gasto (desviación **25.8** puntos) tienen escalas parecidas *por casualidad*: el ingreso viene en **miles** de dólares. ¿Qué pasaría si alguien nos entregara el ingreso **en dólares**? Hagamos el experimento (con 4 clusters, sólo para ilustrar):

In [21]:
# Esta celda va de regalo: léela con calma y ejecútala
# Experimento: el mismo ingreso, pero en dólares en vez de miles de dólares
df_usd = df.assign(ingreso_usd=df['ingreso_anual'] * 1000)
print(df_usd[['ingreso_usd', 'gasto']].std().round(1))

kmeans_usd = KMeans(n_clusters=4, n_init=10, random_state=42).fit(df_usd[['ingreso_usd', 'gasto']])
plot_clusters(df, 'ingreso_anual', 'gasto', kmeans_usd.labels_,
              title='K-Means SIN escalar, con el ingreso en dólares')
perfil_clusters(df, kmeans_usd.labels_, ['ingreso_anual', 'gasto'])


ingreso_usd    26264.7
gasto             25.8
dtype: float64


,n,pct,ingreso_anual,gasto
Cluster 0,50,25.0,27.4,49.5
Cluster 1,62,31.0,77.8,50.4
Cluster 2,68,34.0,54.8,50.0
Cluster 3,20,10.0,109.7,52.0


🚩 **Algo no cuadra:** con el ingreso en dólares (desviación **26,264.7** contra **25.8** del gasto), los clusters son **franjas verticales de ingreso** y el gasto promedio de **todos** los clusters queda alrededor de 50 (**49.5 a 52.0**). El gasto, que es justo lo que más le importa a marketing, **dejó de contar**. Mismos clientes, otra unidad → otros segmentos.

La solución es **estandarizar**: restar la media y dividir entre la desviación estándar, para que cada variable tenga media 0 y desviación 1 y **pese lo mismo**. Eso hace `preparar_mall`. Empezamos con las dos variables de la gráfica: `ingreso_anual` y `gasto`.

In [22]:
COLS_2D = ['ingreso_anual', 'gasto']

# 1. Escala las dos columnas: X_2d, scaler_2d = preparar_mall(df, COLS_2D)
# 2. Imprime la forma de X_2d, y la media y desviación de cada columna:
#    X_2d.mean(axis=0).round(2) y X_2d.std(axis=0).round(2)
# ✍️ Tu código aquí
COLS_2D = ['ingreso_anual', 'gasto']

# Estandarizar las dos columnas
X_2d, scaler_2d = preparar_mall(df, COLS_2D)

# Verificar el resultado
print("Forma de X_2d:", X_2d.shape)
print("Media de cada columna:", X_2d.mean(axis=0).round(2))
print("Desviación de cada columna:", X_2d.std(axis=0).round(2))


Forma de X_2d: (200, 2)
Media de cada columna: [-0. -0.]
Desviación de cada columna: [1. 1.]


✍️ **¿Qué observas?** ¿Qué media y desviación tiene cada columna de `X_2d`? ¿Por qué el experimento con el ingreso en dólares dio clusters tan distintos?

RELLENAR CON TUS COMENTARIOS
-Media de 0 y desviacion de 1.
-Porque el ingreso tiene una mayor desviacion que el gasto. El gasto va de 0 a 100 y el ingreso de 15 k a 140 k

---
# 4️⃣ Modelo 2D: ingreso y gasto, ¿cuántos clusters? (30 min)

K-Means necesita que **tú** le digas k. Ajustamos K-Means para k = 2…10 y miramos dos señales (las viste en los fundamentos):

| Señal | ¿Qué buscamos? |
|---|---|
| Inercia (codo) | Donde la curva deja de bajar mucho |
| Silhouette | El más **alto** |

In [23]:
# Inercia y silhouette para k = 2, ..., 10
# Pista: metricas_2d = tabla_metricas_k(X_2d, range(2, 11)) y muestra la tabla
# ✍️ Tu código aquí
metricas_2d=tabla_metricas_k(X_2d, range(2,11))
metricas_2d



,k,inercia,silhouette
0,2,269.7,0.321
1,3,157.7,0.467
2,4,108.9,0.494
3,5,65.6,0.555
4,6,55.1,0.540
5,7,44.9,0.528
6,8,37.2,0.455
7,9,32.4,0.457
8,10,30.0,0.443


In [28]:
# 1. Método del codo: plot_codo(metricas_2d['k'], metricas_2d['inercia'])
# 2. Silhouette según k: plot_silueta_k(metricas_2d)
# ✍️ Tu código aquí
metricas_2d = tabla_metricas_k(X_2d, range(2, 11))

plot_codo(metricas_2d['k'], metricas_2d['inercia'])
plot_silueta_k(metricas_2d)




La silhouette de la tabla es un promedio. Antes de decidir, miramos la **gráfica de silueta** de los dos candidatos más fuertes:

In [29]:
# Elige los 2 valores de k más prometedores según la tabla y grafica la silueta de cada uno:
# for k in [<k1>, <k2>]:
#     labels_k = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_2d)
#     plot_silueta(X_2d, labels_k, title=f'Silueta con k = {k}')
# ✍️ Tu código aquí
mejores_k = metricas_2d.nlargest(2, 'silhouette')['k'].tolist()
print("Valores de k a comparar:", mejores_k)

for k in mejores_k:
    labels_k = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=42
    ).fit_predict(X_2d)

    plot_silueta(
        X_2d,
        labels_k,
        title=f'Silueta con k = {k}'
    )

Valores de k a comparar: [5, 6]


✍️ **¿Qué observas?**
- ¿Dónde ves el codo?En k = 5, porque después de ese punto la inercia disminuye mucho menos al agregar clusters.
- ¿Qué k tiene la silhouette más alta? ¿Coincide con el codo?k = 5, con 0.555. Sí coincide con el codo.
- En la gráfica de silueta, ¿cuál de tus dos candidatos tiene clusters más parejos?k = 5 presenta tamaños más equilibrados que k = 6, donde aparece un grupo especialmente pequeño.
- **¿Qué k eliges y por qué?** Piensa también en cuántas campañas distintas puede manejar marketing.Elijo cinco clusters porque combinan la mayor silhouette con el codo observado. Además, permiten organizar cinco campañas diferenciadas sin añadir la complejidad de un sexto grupo.

RELLENAR CON TUS COMENTARIOS

In [31]:
# 1. Ajusta el modelo final con el k que elegiste:
#    kmeans_2d = KMeans(n_clusters=<tu k>, n_init=10, random_state=42).fit(X_2d)
# 2. labels_2d = kmeans_2d.labels_
# 3. Imprime cuántos clientes quedaron en cada cluster: np.bincount(labels_2d)
# ✍️ Tu código aquí
kmeans_2d = KMeans(
    n_clusters=5,
    n_init=10,
    random_state=42
).fit(X_2d)

labels_2d = kmeans_2d.labels_

print("Clientes por cluster:", np.bincount(labels_2d))

Clientes por cluster: [81 39 22 35 23]


### 📊 Guardamos los resultados
Igual que en Modelos Predictivos, vamos guardando cada modelo en una lista `comparacion` para la tabla final:

In [32]:
kmeans_2d = KMeans(
    n_clusters=5,
    n_init=10,
    random_state=42
).fit(X_2d)

labels_2d = kmeans_2d.labels_

# Guardar la evaluación para la comparación final
comparacion = []
comparacion.append(
    evaluar_clustering(
        'K-Means 2D (ingreso, gasto)',
        X_2d,
        labels_2d
    )
)

pd.DataFrame(comparacion)

,modelo,n_clusters,pct_ruido,silhouette
0,"K-Means 2D (ingreso, gasto)",5,0.0,0.555


---
# ☕ Descanso (10 min)

---
# 5️⃣ Interpretar los segmentos (25 min)

Un cluster sin nombre no le sirve a nadie. Para nombrarlo necesitamos su **perfil**: cuántos clientes tiene y el promedio de cada variable **en unidades originales**. Incluimos la `edad` aunque no entró al modelo: ayuda a describir a los clientes.

In [ ]:
# Perfil de cada cluster (tamaño y promedios en unidades originales)
# Pista: perfil_2d = perfil_clusters(df, labels_2d, COLS_MALL) y muestra la tabla
# ✍️ Tu código aquí


In [ ]:
# 1. Regresa los centroides a unidades originales:
#    centroides_2d = scaler_2d.inverse_transform(kmeans_2d.cluster_centers_)
# 2. Grafica: plot_clusters(df, 'ingreso_anual', 'gasto', labels_2d, centroides=centroides_2d)
# ✍️ Tu código aquí


✍️ **Ponle nombre a cada segmento.** Completa la tabla con **una fila por cluster** usando tu perfil (tamaño, ingreso, gasto y edad promedio):

| Cluster | Clientes | Perfil (promedios) | Nombre del segmento |
|---|---|---|---|
| ______ | ______ | ______ | ______ |

- ¿Cuál es el segmento más grande? ¿Es el más interesante para marketing?
- La `edad` no entró al modelo: ¿algún segmento es claramente más joven o más grande que los demás?

RELLENAR CON TUS COMENTARIOS

Ya viste la relación edad–gasto en la Parte 2 y, en la Parte 5, la edad promedio de cada segmento. Un joven de 25 años y un adulto de 55 con el mismo ingreso y gasto, ¿responden igual a la misma campaña? Probablemente no. Agreguemos la `edad` al modelo y volvamos a elegir k.

In [ ]:
# 1. Escala las tres variables: X_3d, scaler_3d = preparar_mall(df, COLS_MALL)
# 2. metricas_3d = tabla_metricas_k(X_3d, range(2, 11))
# 3. Grafica el codo y la silhouette (plot_codo y plot_silueta_k) y muestra la tabla
# ✍️ Tu código aquí


✍️ **¿Qué observas?** ¿Se ve un codo claro? ¿Qué k tiene la silhouette más alta? **¿Qué k eliges para el modelo 3D y por qué?** ¿Puedes comparar directamente el silhouette 3D con el del modelo 2D?

RELLENAR CON TUS COMENTARIOS

In [ ]:
# 1. kmeans_3d = KMeans(n_clusters=<tu k>, n_init=10, random_state=42).fit(X_3d)
# 2. labels_3d = kmeans_3d.labels_
# 3. Agrégalo a la comparación: comparacion.append(evaluar_clustering('K-Means 3D (+ edad)', X_3d, labels_3d))
# 4. perfil_3d = perfil_clusters(df, labels_3d, COLS_MALL) y muestra la tabla
# ✍️ Tu código aquí


In [ ]:
# Gráfica 3D (rótala con el mouse)
# Pista: plot_clusters_3d(df, 'edad', 'ingreso_anual', 'gasto', labels_3d)
# ✍️ Tu código aquí


### 🔍 ¿Qué cambió respecto al modelo 2D?
Cruzamos los dos agrupamientos: cada fila es un cluster del modelo 2D y cada columna uno del 3D. Además, para comparar **de forma justa**, evaluamos las etiquetas del modelo 2D en el **mismo espacio 3D**.

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
cruce = pd.crosstab(labels_2d, labels_3d, rownames=['cluster 2D'], colnames=['cluster 3D'])

# Las etiquetas 2D evaluadas con las 3 variables: así sí se comparan con el modelo 3D
comparacion.append(evaluar_clustering('K-Means 2D evaluado en 3D', X_3d, labels_2d))
cruce

✍️ **¿Qué observas?**
- ¿Qué segmentos del modelo 2D se conservan en el 3D? ¿Cuál se partió y en qué se diferencian sus partes?
- Sin contar el segmento que se partió, ¿cuántos clientes cambian de grupo entre un modelo y otro?
- Evaluadas en el mismo espacio 3D, ¿qué silhouette tienen las etiquetas 2D comparado con el modelo 3D?
- Ponle nombre a cada segmento 3D (una fila por cluster):

| Cluster 3D | Clientes | Perfil (promedios) | Nombre del segmento |
|---|---|---|---|
| ______ | ______ | ______ | ______ |

RELLENAR CON TUS COMENTARIOS

---
# 7️⃣ ¿Y el género? (10 min)

Marketing pregunta: *"¿no deberíamos hacer campañas distintas para hombres y mujeres?"*. Veamos si el género **ya** separa los segmentos que encontramos. Compara con la proporción de mujeres en todo el dataset que calculaste en la Parte 1.

In [ ]:
# Proporción de mujeres y hombres dentro de cada cluster 3D
# Pista: pd.crosstab(labels_3d, df['genero'], normalize='index').round(2)
# ✍️ Tu código aquí


¿Y si metemos el género **al modelo**? Lo codificamos como 0/1 (`es_mujer`), lo escalamos junto con las otras tres variables y usamos el mismo k que en el modelo 3D:

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
from sklearn.preprocessing import StandardScaler

df_genero = df.assign(es_mujer=(df['genero'] == 'Female').astype(int))
COLS_GENERO = COLS_MALL + ['es_mujer']
X_genero = StandardScaler().fit_transform(df_genero[COLS_GENERO])

kmeans_genero = KMeans(n_clusters=kmeans_3d.n_clusters, n_init=10, random_state=42).fit(X_genero)
pd.crosstab(kmeans_genero.labels_, df['genero'], rownames=['cluster con género'])

✍️ **¿Qué observas?**
- ¿El porcentaje de mujeres cambia mucho entre clusters comparado con la proporción de todo el dataset?
- Al meter `es_mujer` al modelo, ¿cómo quedan los clusters? ¿Crees que eso es un descubrimiento o un efecto de cómo mide distancias K-Means?
- ¿Dejarías el género dentro o fuera del modelo?

RELLENAR CON TUS COMENTARIOS

---
# 8️⃣ Campañas y conclusiones (20 min)

### 📊 Comparación de modelos

In [ ]:
# Tabla comparativa de los modelos
# Pista: pd.DataFrame(comparacion)
# ✍️ Tu código aquí


✍️ **¿Qué observas?** ¿Qué modelo tiene el silhouette más alto? ¿Es justo comparar el modelo 2D con el 3D usando esa columna? ¿Qué filas sí se pueden comparar entre sí? ¿Con qué modelo te quedas?

RELLENAR CON TUS COMENTARIOS

### 🎯 Una campaña por segmento
✍️ Para cada segmento de tu modelo final propón una campaña concreta (canal, tipo de oferta, objetivo):

| Segmento | Clientes | ¿Quiénes son? | Campaña |
|---|---|---|---|
| ______ | ______ | ______ | ______ |

## ❓ Respuesta a la pregunta
> *¿Qué segmentos de clientes tiene el centro comercial y qué campaña le toca a cada uno?*

✍️ Responde con tus resultados: ¿cuántos segmentos encontraste?, ¿cuáles son los más importantes para el negocio?, ¿sirve el género para segmentar?

RELLENAR CON TUS COMENTARIOS

## 📝 Lo que aprendimos
✍️ Escribe 3 cosas que aprendiste hoy.

RELLENAR CON TUS COMENTARIOS

## 🚀 Retos opcionales
- Ajusta el modelo 3D con **un k menos** y **un k más** que el que elegiste. ¿Qué segmento aparece o desaparece? ¿Cuál le sirve más a marketing?
- Repite el modelo 2D con `MinMaxScaler` en lugar de `StandardScaler`. ¿Cambian los segmentos? (Pista: cruza las etiquetas de ambos modelos con `pd.crosstab` y cuenta cuántos clientes cambian de segmento.)
- Agrupa sólo con **edad y gasto**. ¿Qué segmentos salen? ¿Son útiles?
- 🔜 **Siguiente sesión:** ¿hay clientes que **no encajan** en ningún segmento? K-Means obliga a todos a pertenecer a un cluster; **DBSCAN** puede marcarlos como ruido.